# Simulación Computacional: Fisica Molecular
**Área:** Fisica Atomica y Molecular

Este cuaderno interactivo de Jupyter ejecuta numéricamente los modelos físicos descritos en el repositorio. Puedes modificar los parámetros iniciales y ejecutar las celdas para visualizar gráficos o animaciones interactivos.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import eigsh

# Parámetros para un oscilador molecular de Morse genérico (ej. H2)
De = 4.75          # Energía de disociación en eV
re = 0.74          # Distancia internuclear de equilibrio en Angstroms
a = 1.93           # Parámetro de anchura del pozo en Angstrom^-1
mu = 918.0         # Masa reducida relativa a la masa electrónica
hbar2_over_2m = 3.81  # Factor de conversión (aprox) para energías en eV y dist en A

# Grid espacial (Distancia radial internuclear R)
N = 1000
R = np.linspace(0.2, 4.0, N)
dR = R[1] - R[0]

# Superficie de Energía Potencial (PES): Potencial de Morse
V_morse = De * (1.0 - np.exp(-a * (R - re)))**2 - De

# Hamiltoniano mediante diferencias finitas
t_coeff = -hbar2_over_2m / (mu * dR**2)
main_diag = -2.0 * t_coeff * np.ones(N) + V_morse
off_diag = t_coeff * np.ones(N-1)

H = diags([off_diag, main_diag, off_diag], [-1, 0, 1], format='csc')

# Resolviendo los autovalores y autofunciones
num_levels = 8
eigenvalues, eigenvectors = eigsh(H, k=num_levels, which='SM')

plt.figure(figsize=(10, 6))
plt.plot(R, V_morse, 'k-', lw=2, label='Potencial de Morse V(R)')
plt.axhline(0, color='gray', linestyle='--', label='Límite de Disociación')

scale = 0.5
colors = plt.cm.viridis(np.linspace(0, 1, num_levels))

for i in range(num_levels):
    E = eigenvalues[i]
    psi = eigenvectors[:, i]
    if psi[np.argmax(np.abs(psi))] < 0:
        psi = -psi
    
    plt.axhline(E, color=colors[i], linestyle=':', alpha=0.5)
    plt.plot(R, E + scale * psi, color=colors[i], label=f'v={i}')

plt.title("Niveles Vibracionales Anarmónicos (Potencial de Morse)")
plt.xlabel("Distancia Internuclear $R$ (Å)")
plt.ylabel("Energía (eV)")
plt.ylim(-De - 0.5, 1.0)
plt.xlim(0.2, 4.0)
plt.legend(loc='lower right', ncol=2)
plt.grid(True, alpha=0.2)
plt.tight_layout()
# plt.show()